# Agentic AI for Bankers — SFI Masterclass

> **Goal:** Build a fully autonomous AI agent that reads, searches, and reasons over real SEC 10-K filings from the S&P 100 — no API keys required.

---

### What you will learn

| Concept | What it means for banking |
|---|---|
| **Retrieval-Augmented Generation (RAG)** | Ground AI answers in *actual* regulatory filings — not hallucinations |
| **Chunking & embedding** | Turn 200-page 10-Ks into a searchable vector database |
| **Agentic loops** | Let the AI *plan*, *retrieve*, *reflect*, and *iterate* — like a junior analyst |
| **Tool use** | Give the agent access to structured "tools" (search, compare, extract) |

### Data

We have pre-downloaded the **most recent 5 years of 10-K annual reports** for all **100 S&P 100 companies** straight from SEC EDGAR. The filings are hosted on GitHub for instant access — **no SEC rate-limiting, no API keys, no authentication**.

```
https://raw.githubusercontent.com/SMalamud/sec-10k-downloader/main/10k_filings/{TICKER}/10K_{DATE}.htm
```

Let's get started.

---
## 1 — Environment Setup

We install only lightweight, open-source packages. Everything runs on a free Colab CPU runtime.

| Package | Role |
|---|---|
| `beautifulsoup4` / `lxml` | Parse raw HTML from 10-K filings into clean text |
| `scikit-learn` | TF-IDF vectoriser (our "embedding" model — zero cost) |
| `faiss-cpu` | Facebook's billion-scale similarity search — here used for instant retrieval |
| `tqdm` | Progress bars for long loops |

In [ ]:
!pip -q install pandas numpy requests beautifulsoup4 lxml tqdm scikit-learn faiss-cpu

---
## 2 — Imports

In [ ]:
import re, json, textwrap
from collections import defaultdict

import requests
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import TfidfVectorizer
import faiss
from tqdm.auto import tqdm

---
## 3 — Load 10-K Filings from GitHub

All filings are pre-downloaded and hosted publicly. We fetch them over HTTPS — **no SEC credentials, no rate limits**.

**Architecture choice:** In production you would use a database. Here we keep everything in memory so the notebook stays self-contained.

### 3.1 — Catalogue of available filings

We first pull the repo's directory listing to discover every `TICKER/10K_DATE.htm` file.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
REPO_BASE = "https://raw.githubusercontent.com/SMalamud/sec-10k-downloader/main/10k_filings"
GITHUB_API = "https://api.github.com/repos/SMalamud/sec-10k-downloader/contents/10k_filings"

# S&P 100 tickers
SP100 = [
    "AAPL","ABBV","ABT","ACN","ADBE","AIG","AMD","AMGN","AMT","AMZN",
    "AVGO","AXP","BA","BAC","BK","BKNG","BLK","BMY","BRK.B","C",
    "CAT","CHTR","CL","CMCSA","COF","COP","COST","CRM","CSCO","CVS",
    "CVX","DE","DHR","DIS","DOW","DUK","EMR","EXC","F","FDX",
    "GD","GE","GILD","GM","GOOG","GS","HD","HON","IBM","INTC",
    "INTU","JNJ","JPM","KHC","KO","LIN","LLY","LMT","LOW","MA",
    "MCD","MDLZ","MDT","MET","META","MMM","MO","MRK","MS","MSFT",
    "NEE","NFLX","NKE","NVDA","ORCL","PEP","PFE","PG","PM","PYPL",
    "QCOM","RTX","SBUX","SCHW","SO","SPG","T","TGT","TMO","TMUS",
    "TXN","UNH","UNP","UPS","USB","V","VZ","WFC","WMT","XOM",
]

def list_filings(ticker):
    """Return list of (ticker, date, url) for all available 10-Ks."""
    api_url = f"{GITHUB_API}/{ticker}"
    resp = requests.get(api_url)
    if resp.status_code != 200:
        return []
    files = resp.json()
    results = []
    for f in files:
        name = f["name"]
        m = re.match(r"10K_(\d{4}-\d{2}-\d{2})\.htm", name)
        if m:
            results.append((ticker, m.group(1), f"{REPO_BASE}/{ticker}/{name}"))
    return sorted(results, key=lambda x: x[1], reverse=True)

# Quick test — show AAPL filings
list_filings("AAPL")

### 3.2 — HTML → Clean text

10-K filings are filed as HTML. We need to:
1. **Strip** JavaScript, CSS, and formatting tags
2. **Collapse** whitespace
3. Return pure text that we can chunk and search

This is the same pre-processing pipeline every production RAG system uses.

In [ ]:
def fetch_10k_text(url):
    """Download a 10-K HTML file and return clean text."""
    html = requests.get(url).text
    soup = BeautifulSoup(html, "lxml")
    for tag in soup(["script", "style", "header", "footer", "nav"]):
        tag.decompose()
    text = soup.get_text(separator=" ")
    return re.sub(r"\s+", " ", text).strip()

# Quick test — first 500 chars of Apple's latest 10-K
sample_url = f"{REPO_BASE}/AAPL/10K_2025-10-31.htm"
sample_text = fetch_10k_text(sample_url)
print(f"Length: {len(sample_text):,} characters")
print(sample_text[:500])

### 3.3 — Chunking: turning documents into searchable pieces

A single 10-K can be 80,000+ words. No retrieval system works well on documents that large.

**Chunking** splits each filing into overlapping windows of ~500 words. This is a fundamental design choice in every RAG system:

| Parameter | Trade-off |
|---|---|
| **Chunk size** | Larger = more context per chunk, but noisier retrieval |
| **Overlap** | Prevents important sentences from being split across chunks |

We use **500 words with 50-word overlap** — a solid default for financial text.

In [ ]:
CHUNK_SIZE = 500    # words per chunk
CHUNK_OVERLAP = 50  # overlapping words between consecutive chunks

def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """Split text into overlapping word-windows."""
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i : i + chunk_size])
        if len(chunk.split()) > 20:  # skip tiny trailing chunks
            chunks.append(chunk)
    return chunks

# Demo: chunk the AAPL sample
demo_chunks = chunk_text(sample_text)
print(f"AAPL latest 10-K: {len(sample_text.split()):,} words → {len(demo_chunks)} chunks")

### 3.4 — Build the corpus

We load the **most recent 10-K** for a selection of major companies spanning different sectors. (Loading all 100 companies x 5 years is ~460 filings — feel free to expand the list!)

| Sector | Tickers |
|---|---|
| Big Tech | AAPL, MSFT, GOOG, AMZN, META, NVDA |
| Banking & Finance | JPM, GS, BAC, MS, BLK, MA |
| Healthcare & Pharma | JNJ, PFE, UNH, LLY, ABBV, MRK |
| Energy | XOM, CVX, COP |
| Consumer | WMT, KO, MCD, NKE |
| Industrial | CAT, BA, GE, HON |

In [ ]:
# ── Latest filing dates (hardcoded to avoid GitHub API rate limits) ──────────
LATEST_FILINGS = {
    # Tech
    "AAPL": "2025-10-31", "MSFT": "2025-07-30", "GOOG": "2026-02-05",
    "AMZN": "2026-02-06", "META": "2026-01-29", "NVDA": "2025-02-26",
    # Banking & Finance
    "JPM": "2026-02-13", "GS": "2025-02-27", "BAC": "2025-02-25",
    "MS": "2025-02-21", "BLK": "2025-02-25", "MA": "2026-02-11",
    # Healthcare
    "JNJ": "2026-02-11", "PFE": "2025-02-27", "UNH": "2025-02-27",
    "LLY": "2026-02-12", "ABBV": "2025-02-14", "MRK": "2025-02-25",
    # Energy
    "XOM": "2025-02-19", "CVX": "2025-02-21", "COP": "2025-02-18",
    # Consumer
    "WMT": "2025-03-14", "KO": "2025-02-20", "MCD": "2025-02-25", "NKE": "2025-07-17",
    # Industrial
    "CAT": "2026-02-13", "BA": "2026-01-30", "GE": "2026-01-29", "HON": "2025-02-14",
}
TICKERS_TO_LOAD = list(LATEST_FILINGS.keys())

# ── For each ticker, fetch the most recent 10-K directly (no API calls) ─────
chunks = []     # list of text chunks
metadata = []   # parallel list: {"ticker": ..., "date": ..., "chunk_idx": ...}

for ticker in tqdm(TICKERS_TO_LOAD, desc="Loading 10-Ks"):
    date = LATEST_FILINGS[ticker]
    url = f"{REPO_BASE}/{ticker}/10K_{date}.htm"
    text = fetch_10k_text(url)
    if len(text) < 100:
        print(f"  SKIP {ticker}: empty or failed download")
        continue
    ticker_chunks = chunk_text(text)
    for i, c in enumerate(ticker_chunks):
        chunks.append(c)
        metadata.append({"ticker": ticker, "date": date, "chunk_idx": i})

print(f"\nCorpus: {len(TICKERS_TO_LOAD)} companies → {len(chunks):,} chunks")

---
## 4 — Vector Index (the "brain" of RAG)

### How RAG works — the banker's version

Imagine you're a junior analyst with 30 filing binders on your desk. Someone asks: *"Which companies mention supply-chain risk from China?"*

You **don't** read all 30 binders cover-to-cover. Instead you:
1. Think of **keywords** that would appear near the answer
2. **Flip** to the most relevant pages
3. **Read** those pages carefully
4. **Synthesise** an answer

RAG does exactly this, but mathematically:

| Analyst action | RAG equivalent |
|---|---|
| Think of keywords | Encode the query as a vector |
| Flip to relevant pages | Nearest-neighbour search in vector space |
| Read carefully | Feed retrieved chunks to the LLM |
| Synthesise | LLM generates a grounded answer |

### Our approach: TF-IDF + FAISS

- **TF-IDF** (Term Frequency–Inverse Document Frequency) converts text into vectors where rare, informative words get high weight
- **FAISS** (Facebook AI Similarity Search) builds an index for instant cosine-similarity lookup

This is a *zero-cost* alternative to OpenAI embeddings — and surprisingly effective on domain-specific financial text.

In [ ]:
# ── Build TF-IDF vectors ────────────────────────────────────────────────────
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=25_000,   # keep top 25k terms
    ngram_range=(1, 2),    # unigrams + bigrams for better phrase matching
)

X = vectorizer.fit_transform(chunks).toarray().astype("float32")
faiss.normalize_L2(X)  # normalize so dot-product = cosine similarity

# ── Build FAISS index ───────────────────────────────────────────────────────
index = faiss.IndexFlatIP(X.shape[1])   # Inner Product (= cosine on normalised vecs)
index.add(X)

print(f"Index built: {index.ntotal:,} vectors, {X.shape[1]:,} dimensions")

In [ ]:
def retrieve(query, k=5):
    """Search the vector index. Returns list of (chunk_text, metadata, score)."""
    q_vec = vectorizer.transform([query]).toarray().astype("float32")
    faiss.normalize_L2(q_vec)
    scores, indices = index.search(q_vec, k)
    results = []
    for rank, idx in enumerate(indices[0]):
        if idx < 0:
            continue
        results.append({
            "rank": rank + 1,
            "ticker": metadata[idx]["ticker"],
            "date": metadata[idx]["date"],
            "score": float(scores[0][rank]),
            "text": chunks[idx][:300] + "...",   # preview
        })
    return results

# ── Test retrieval ──────────────────────────────────────────────────────────
results = retrieve("artificial intelligence risk and regulation")
pd.DataFrame(results)[["rank", "ticker", "date", "score", "text"]]

---
## 5 — The Agentic Loop

### What makes an agent different from a chatbot?

A **chatbot** answers questions in one shot. An **agent** breaks a complex goal into steps and executes them autonomously:

```
┌─────────────────────────────────────────────────────────────┐
│                    AGENTIC LOOP                             │
│                                                             │
│  GOAL ──→ PLAN ──→ ACT ──→ OBSERVE ──→ REFLECT ──→ PLAN   │
│              │                              │               │
│              └──────────────────────────────┘               │
│                   (repeat until done)                       │
└─────────────────────────────────────────────────────────────┘
```

### Why does this matter for banking?

| Traditional automation | Agentic AI |
|---|---|
| Pre-coded rules, brittle | Dynamic planning, adaptive |
| One task at a time | Decomposes complex goals |
| Fails on ambiguity | Asks clarifying sub-questions |
| No memory across steps | Accumulates context in a scratchpad |

### Our agent's "tools"

We give the agent access to three tools — just like giving an analyst access to Bloomberg, EDGAR, and a calculator:

| Tool | Description |
|---|---|
| `search(query)` | Semantic search over all 10-K chunks |
| `compare(ticker1, ticker2, topic)` | Side-by-side comparison on a topic |
| `extract_risks(ticker)` | Find and list risk factors for a company |

In [ ]:
# ── Agent Tools ──────────────────────────────────────────────────────────────

def tool_search(query, k=5):
    """TOOL: Semantic search across all loaded 10-K filings."""
    q_vec = vectorizer.transform([query]).toarray().astype("float32")
    faiss.normalize_L2(q_vec)
    scores, indices = index.search(q_vec, k)
    results = []
    for rank, idx in enumerate(indices[0]):
        if idx < 0:
            continue
        results.append({
            "ticker": metadata[idx]["ticker"],
            "date": metadata[idx]["date"],
            "score": round(float(scores[0][rank]), 4),
            "excerpt": chunks[idx][:500],
        })
    return results


def tool_compare(ticker1, ticker2, topic, k=3):
    """TOOL: Side-by-side comparison of two companies on a specific topic."""
    results = {"ticker1": ticker1, "ticker2": ticker2, "topic": topic}
    for ticker in [ticker1, ticker2]:
        query = f"{ticker} {topic}"
        q_vec = vectorizer.transform([query]).toarray().astype("float32")
        faiss.normalize_L2(q_vec)
        scores, indices = index.search(q_vec, k)
        excerpts = []
        for rank, idx in enumerate(indices[0]):
            if idx >= 0 and metadata[idx]["ticker"] == ticker:
                excerpts.append(chunks[idx][:400])
        # If no ticker-specific match, take best overall
        if not excerpts:
            for rank, idx in enumerate(indices[0]):
                if idx >= 0:
                    excerpts.append(f"[{metadata[idx]['ticker']}] {chunks[idx][:400]}")
        results[ticker] = excerpts[:k]
    return results


def tool_extract_risks(ticker, k=8):
    """TOOL: Extract risk-related passages for a specific company."""
    query = f"{ticker} risk factors material risks uncertainties"
    q_vec = vectorizer.transform([query]).toarray().astype("float32")
    faiss.normalize_L2(q_vec)
    scores, indices = index.search(q_vec, k * 2)  # fetch extra, then filter
    risks = []
    for rank, idx in enumerate(indices[0]):
        if idx >= 0 and metadata[idx]["ticker"] == ticker:
            risks.append({
                "score": round(float(scores[0][rank]), 4),
                "excerpt": chunks[idx][:600],
            })
        if len(risks) >= k:
            break
    return {"ticker": ticker, "num_risk_passages": len(risks), "risks": risks}


TOOLS = {
    "search": tool_search,
    "compare": tool_compare,
    "extract_risks": tool_extract_risks,
}

print("Agent tools registered:", list(TOOLS.keys()))

### 5.1 — The Agent class

Below is a **rule-based agent** that demonstrates the core agentic pattern. In production, the planner would be an LLM (GPT-4, Claude, etc.) that decides which tools to call. Here we hard-code the planning logic so everything runs without API keys.

The key insight: **the agent maintains a scratchpad** — a running memory of what it has done and found. Each step builds on previous observations. This is what distinguishes agentic AI from simple function chaining.

In [ ]:
class FinancialAgent:
    """
    A simple agentic loop over 10-K filings.
    
    Architecture:
        GOAL → PLAN → [ACT → OBSERVE → REFLECT]* → FINAL ANSWER
    
    In production, each step would be an LLM call.
    Here we use deterministic rules to demonstrate the pattern.
    """
    
    def __init__(self, tools):
        self.tools = tools
        self.scratchpad = []     # running memory of (action, observation) pairs
        self.plan = []           # current plan steps
    
    def _log(self, role, msg):
        """Pretty-print agent reasoning."""
        icons = {"PLAN": "📋", "ACT": "🔧", "OBSERVE": "👁️", "REFLECT": "🤔", "ANSWER": "✅"}
        icon = icons.get(role, "•")
        print(f"\n{icon}  [{role}] {msg}")
    
    def _plan(self, goal):
        """PLAN: Decompose a goal into sub-tasks (rule-based planner)."""
        self.plan = []
        goal_lower = goal.lower()
        
        # Pattern: comparison questions
        if "compare" in goal_lower or " vs " in goal_lower or "versus" in goal_lower:
            # Extract tickers mentioned in the goal
            mentioned = [t for t in TICKERS_TO_LOAD if t.lower() in goal_lower or t in goal]
            if len(mentioned) >= 2:
                topic = goal_lower.replace("compare", "").replace(mentioned[0].lower(), "").replace(mentioned[1].lower(), "").strip()
                self.plan = [
                    ("extract_risks", {"ticker": mentioned[0]}),
                    ("extract_risks", {"ticker": mentioned[1]}),
                    ("compare", {"ticker1": mentioned[0], "ticker2": mentioned[1], "topic": topic or "business strategy"}),
                ]
            else:
                self.plan = [("search", {"query": goal})]
        
        # Pattern: risk-focused questions
        elif "risk" in goal_lower:
            mentioned = [t for t in TICKERS_TO_LOAD if t.lower() in goal_lower or t in goal]
            if mentioned:
                for t in mentioned[:3]:
                    self.plan.append(("extract_risks", {"ticker": t}))
            else:
                self.plan = [
                    ("search", {"query": goal}),
                    ("search", {"query": f"risk factors {goal}"}),
                ]
        
        # Pattern: broad search questions
        else:
            self.plan = [
                ("search", {"query": goal}),
                ("search", {"query": f"financial performance {goal}"}),
            ]
        
        self._log("PLAN", f"Decomposed into {len(self.plan)} steps:")
        for i, (tool, args) in enumerate(self.plan):
            self._log("PLAN", f"  Step {i+1}: {tool}({args})")
        return self.plan
    
    def _act(self, tool_name, tool_args):
        """ACT: Execute a tool."""
        self._log("ACT", f"Calling {tool_name}({tool_args})")
        result = self.tools[tool_name](**tool_args)
        return result
    
    def _observe(self, result):
        """OBSERVE: Summarise what we found."""
        if isinstance(result, list):
            tickers_found = list({r.get("ticker", "?") for r in result})
            summary = f"Found {len(result)} results from: {', '.join(tickers_found)}"
        elif isinstance(result, dict) and "risks" in result:
            summary = f"Extracted {result['num_risk_passages']} risk passages for {result['ticker']}"
        elif isinstance(result, dict) and "topic" in result:
            summary = f"Compared {result['ticker1']} vs {result['ticker2']} on '{result['topic']}'"
        else:
            summary = f"Got result of type {type(result).__name__}"
        
        self._log("OBSERVE", summary)
        return summary
    
    def _reflect(self):
        """REFLECT: Review accumulated findings."""
        n_observations = len(self.scratchpad)
        tickers_covered = list({
            obs.get("ticker", "")
            for _, obs_list in self.scratchpad
            if isinstance(obs_list, list)
            for obs in obs_list
            if isinstance(obs, dict)
        })
        self._log("REFLECT", 
            f"After {n_observations} steps, covered tickers: "
            f"{', '.join(tickers_covered) if tickers_covered else 'various'}")
    
    def run(self, goal):
        """Execute the full agentic loop."""
        print("=" * 70)
        self._log("PLAN", f'GOAL: "{goal}"')
        print("=" * 70)
        
        self.scratchpad = []
        self._plan(goal)
        
        for step_num, (tool_name, tool_args) in enumerate(self.plan):
            result = self._act(tool_name, tool_args)
            summary = self._observe(result)
            self.scratchpad.append((summary, result))
        
        self._reflect()
        
        # Compile final answer
        self._log("ANSWER", "Compiling findings...")
        return self._compile_answer(goal)
    
    def _compile_answer(self, goal):
        """Compile all observations into a structured answer."""
        answer = {"goal": goal, "steps_executed": len(self.scratchpad), "findings": []}
        
        for summary, result in self.scratchpad:
            if isinstance(result, list):
                for r in result[:3]:
                    answer["findings"].append({
                        "ticker": r.get("ticker"),
                        "score": r.get("score"),
                        "excerpt": r.get("excerpt", "")[:300],
                    })
            elif isinstance(result, dict) and "risks" in result:
                for r in result["risks"][:3]:
                    answer["findings"].append({
                        "ticker": result["ticker"],
                        "type": "risk_factor",
                        "score": r["score"],
                        "excerpt": r["excerpt"][:300],
                    })
            elif isinstance(result, dict) and "topic" in result:
                answer["findings"].append({
                    "type": "comparison",
                    "topic": result["topic"],
                    "ticker1": result["ticker1"],
                    "ticker2": result["ticker2"],
                })
        
        return answer

agent = FinancialAgent(tools=TOOLS)
print("Agent initialised with tools:", list(TOOLS.keys()))

---
## 6 — Run the Agent

Let's put the agent through its paces with three real-world banking questions.

### Demo 1: AI risk landscape across sectors
*"Which companies discuss AI-related risks in their latest 10-K?"*

In [ ]:
answer = agent.run("What are the main artificial intelligence risks disclosed by tech companies?")
pd.DataFrame(answer["findings"])

### Demo 2: Head-to-head comparison
*"Compare JPM vs GS on their approach to credit risk."*

In [ ]:
answer = agent.run("Compare JPM vs GS on credit risk and loan losses")
pd.DataFrame(answer["findings"])

### Demo 3: Sector-wide risk extraction
*"What are the key risk factors for energy companies?"*

In [ ]:
answer = agent.run("What are the climate and environmental risk factors for XOM, CVX, and COP?")
pd.DataFrame(answer["findings"])

---
## 7 — Visualising the Vector Space

Let's peek under the hood. We can project our high-dimensional TF-IDF vectors down to 2D to see how the filing chunks cluster by company and sector.

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Sector mapping for colouring
SECTORS = {
    "Tech": ["AAPL","MSFT","GOOG","AMZN","META","NVDA"],
    "Banking": ["JPM","GS","BAC","MS","BLK","MA"],
    "Healthcare": ["JNJ","PFE","UNH","LLY","ABBV","MRK"],
    "Energy": ["XOM","CVX","COP"],
    "Consumer": ["WMT","KO","MCD","NKE"],
    "Industrial": ["CAT","BA","GE","HON"],
}
ticker_to_sector = {t: s for s, tickers in SECTORS.items() for t in tickers}

# PCA → 2D
pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X)

# Plot
fig, ax = plt.subplots(figsize=(12, 8))
colors = {"Tech": "#1f77b4", "Banking": "#ff7f0e", "Healthcare": "#2ca02c",
          "Energy": "#d62728", "Consumer": "#9467bd", "Industrial": "#8c564b"}

for sector, color in colors.items():
    mask = [ticker_to_sector.get(m["ticker"]) == sector for m in metadata]
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1], c=color, label=sector,
               alpha=0.4, s=10)

ax.legend(fontsize=11, markerscale=3)
ax.set_title("10-K Filing Chunks in Vector Space (PCA projection)", fontsize=14)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
plt.tight_layout()
plt.show()

print("Notice how banking filings cluster together — they share regulatory language "
      "that is very different from tech or energy filings.")

---
## 8 — Cross-Company Risk Heatmap

Which topics appear most in which companies' risk disclosures? Let's build a heatmap by searching for key banking risk themes across all loaded companies.

In [ ]:
# ── Risk themes to scan for ──────────────────────────────────────────────────
RISK_THEMES = [
    "artificial intelligence machine learning",
    "cybersecurity data breach",
    "climate change environmental",
    "interest rate monetary policy",
    "supply chain disruption",
    "regulatory compliance government",
    "geopolitical trade tariff",
    "inflation cost pressure",
    "credit risk loan losses",
    "competition market share",
]

# ── For each theme, search and record which tickers appear ──────────────────
heatmap_data = defaultdict(lambda: defaultdict(float))

for theme in tqdm(RISK_THEMES, desc="Scanning risk themes"):
    results = tool_search(theme, k=20)
    for r in results:
        heatmap_data[theme.split()[0] + "_" + theme.split()[1]][r["ticker"]] += r["score"]

# ── Build DataFrame and plot ────────────────────────────────────────────────
df_heat = pd.DataFrame(heatmap_data).fillna(0)
df_heat = df_heat.loc[df_heat.sum(axis=1).sort_values(ascending=False).index]  # sort rows

fig, ax = plt.subplots(figsize=(14, 8))
im = ax.imshow(df_heat.values, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(len(df_heat.columns)))
ax.set_xticklabels(df_heat.columns, rotation=45, ha="right", fontsize=10)
ax.set_yticks(range(len(df_heat.index)))
ax.set_yticklabels(df_heat.index, fontsize=11)
ax.set_title("Risk Theme Relevance by Company (TF-IDF similarity scores)", fontsize=13)
fig.colorbar(im, ax=ax, shrink=0.8, label="Cumulative similarity")
plt.tight_layout()
plt.show()

---
## 9 — Recap: What We Built So Far (Zero-Cost Version)

Everything above runs **for free** — no API keys, no GPU, no paid services. That's great for learning, but in production you want:

- **Better embeddings** — OpenAI's `text-embedding-3-small` captures semantic meaning far better than TF-IDF
- **LLM-powered retrieval** — the model can *reason* about what to search for, not just match keywords
- **LLM-powered planning** — instead of hard-coded `if/else` rules, GPT-4o decides which tools to call
- **Natural-language answers** — the LLM synthesises a polished answer from retrieved chunks

The next sections show how to upgrade each component using the **OpenAI API**.

---
# Part II — Production-Grade RAG with OpenAI

> **Requires:** An OpenAI API key. If you don't have one, you can read along — the code is fully self-contained and the outputs are illustrative.

---
## 10 — Setup: OpenAI API Key

Paste your key below or set it as a Colab secret (`OPENAI_API_KEY`).

**Cost estimate:** Embedding 5,000 chunks with `text-embedding-3-small` costs ~$0.01. Each GPT-4o agent call costs ~$0.01–0.05. The entire notebook should cost **under $1**.

In [ ]:
!pip -q install openai

import os

try:
    from openai import OpenAI
    _openai_installed = True
except ImportError:
    _openai_installed = False
    print("openai package not installed. Run: pip install openai")

# ── Set your API key ────────────────────────────────────────────────────────
# Option 1: Paste directly (not recommended for shared notebooks)
# os.environ["OPENAI_API_KEY"] = "sk-..."

# Option 2: Colab secrets (recommended)
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    pass

OPENAI_AVAILABLE = _openai_installed and bool(os.environ.get("OPENAI_API_KEY"))

if OPENAI_AVAILABLE:
    oai = OpenAI()
    print("OpenAI client ready.")
else:
    oai = None
    if _openai_installed:
        print("No OPENAI_API_KEY found. Set it above to run Part II.")
    else:
        print("Part II cells will be skipped (openai not installed).")

---
## 11 — OpenAI Embeddings: Replacing TF-IDF

### TF-IDF vs Neural Embeddings

| | TF-IDF | OpenAI `text-embedding-3-small` |
|---|---|---|
| **How it works** | Counts word frequencies, weights by rarity | Neural network trained on billions of text pairs |
| **Understands synonyms?** | No — "revenue" and "sales" are different tokens | Yes — maps them to nearby vectors |
| **Understands context?** | No — bag of words, order doesn't matter | Yes — "bank" (finance) vs "bank" (river) |
| **Dimensions** | 25,000 (sparse) | 1,536 (dense) |
| **Cost** | Free | ~$0.02 per 1M tokens |
| **Best for** | Keyword-heavy domains, prototyping | Production systems, semantic search |

### Why this matters for banking

Financial filings use inconsistent terminology: "provisions for credit losses", "allowance for loan losses", "expected credit losses" all mean roughly the same thing. TF-IDF treats them as different words. Neural embeddings understand they're semantically similar.

In [ ]:
EMBED_MODEL = "text-embedding-3-small"
EMBED_DIMS = 1536

def get_embeddings(texts, model=EMBED_MODEL):
    """Get OpenAI embeddings for a list of texts. Batches automatically."""
    all_embeddings = []
    batch_size = 100  # OpenAI allows up to 2048, but 100 is safer for large chunks
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding"):
        batch = texts[i : i + batch_size]
        resp = oai.embeddings.create(input=batch, model=model)
        all_embeddings.extend([e.embedding for e in resp.data])
    return np.array(all_embeddings, dtype="float32")

if OPENAI_AVAILABLE:
    # Embed all chunks (reuses the same `chunks` list from Part I)
    X_oai = get_embeddings(chunks)
    faiss.normalize_L2(X_oai)

    # Build FAISS index
    index_oai = faiss.IndexFlatIP(EMBED_DIMS)
    index_oai.add(X_oai)

    print(f"OpenAI index built: {index_oai.ntotal:,} vectors, {EMBED_DIMS} dimensions")
else:
    print("Skipping — no OpenAI API key. Set OPENAI_API_KEY above to run this cell.")

In [ ]:
def retrieve_oai(query, k=5):
    """Semantic search using OpenAI embeddings."""
    q_vec = get_embeddings([query])
    faiss.normalize_L2(q_vec)
    scores, indices = index_oai.search(q_vec, k)
    results = []
    for rank, idx in enumerate(indices[0]):
        if idx < 0:
            continue
        results.append({
            "rank": rank + 1,
            "ticker": metadata[idx]["ticker"],
            "date": metadata[idx]["date"],
            "score": round(float(scores[0][rank]), 4),
            "text": chunks[idx][:300] + "...",
        })
    return results

if OPENAI_AVAILABLE:
    # Compare: same query, TF-IDF vs OpenAI
    query = "provisions for credit losses and loan write-offs"

    print("=" * 70)
    print(f"Query: \"{query}\"")
    print("=" * 70)

    print("\n--- TF-IDF results ---")
    for r in retrieve(query, k=3):
        print(f"  {r['ticker']:6s} (score={r['score']:.4f})")

    print("\n--- OpenAI embedding results ---")
    for r in retrieve_oai(query, k=3):
        print(f"  {r['ticker']:6s} (score={r['score']:.4f})")

    print("\nNotice how OpenAI embeddings may surface different (often more relevant) "
          "results because they understand synonyms and financial context.")
else:
    print("Skipping — no OpenAI API key.")

---
## 12 — LLM-Powered RAG: Ask GPT-4o Questions Grounded in Real Filings

This is the **production RAG pattern** used at banks, law firms, and consulting firms today:

```
User question
     │
     ▼
┌──────────────┐     ┌──────────────┐     ┌──────────────┐
│  1. Embed    │ ──→ │  2. Retrieve │ ──→ │  3. Generate │
│  the query   │     │  top-k chunks│     │  answer with │
│  (OpenAI)    │     │  (FAISS)     │     │  GPT-4o      │
└──────────────┘     └──────────────┘     └──────────────┘
```

The key: GPT-4o **never sees the full filing**. It only sees the 5 most relevant chunks. This keeps costs low and prevents hallucination.

In [ ]:
LLM_MODEL = "gpt-4o"

def rag_query(question, k=5):
    """
    Full RAG pipeline:
      1. Embed the question
      2. Retrieve top-k chunks from the FAISS index
      3. Send chunks + question to GPT-4o for a grounded answer
    """
    # Step 1 & 2: Retrieve
    results = retrieve_oai(question, k=k)

    # Build context from retrieved chunks
    context_parts = []
    for r in results:
        full_chunk = chunks[
            next(i for i, m in enumerate(metadata)
                 if m["ticker"] == r["ticker"] and m["date"] == r["date"]
                 and chunks[i][:100] == r["text"][:100])
        ]
        context_parts.append(
            f"[{r['ticker']} | 10-K filed {r['date']} | relevance={r['score']:.3f}]\n{full_chunk}"
        )
    context = "\n\n---\n\n".join(context_parts)

    # Step 3: Generate
    system_prompt = """You are a senior financial analyst assistant. Answer the user's question
based ONLY on the provided excerpts from SEC 10-K filings. 

Rules:
- Cite which company (ticker) and filing date each fact comes from
- If the excerpts don't contain enough information, say so explicitly
- Be specific: quote numbers, percentages, and dollar amounts when available
- Structure your answer with clear sections if comparing multiple companies"""

    response = oai.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"## Retrieved Filing Excerpts\n\n{context}\n\n## Question\n\n{question}"},
        ],
        temperature=0.2,
        max_tokens=1500,
    )

    answer = response.choices[0].message.content
    cost_input = response.usage.prompt_tokens
    cost_output = response.usage.completion_tokens

    return {
        "question": question,
        "answer": answer,
        "sources": [(r["ticker"], r["date"], r["score"]) for r in results],
        "tokens": {"input": cost_input, "output": cost_output},
    }

print("rag_query() ready.")

### 12.1 — RAG Demo: Natural-language Q&A over 10-K filings

In [ ]:
if OPENAI_AVAILABLE:
    result = rag_query("How do JPMorgan and Goldman Sachs differ in their approach to credit risk management?")

    print("QUESTION:", result["question"])
    print(f"\nSOURCES: {', '.join(f'{t} ({d})' for t, d, s in result['sources'])}")
    print(f"TOKENS:  {result['tokens']['input']} in, {result['tokens']['output']} out")
    print("\n" + "=" * 70)
    print(result["answer"])
else:
    print("Skipping — no OpenAI API key.")

In [ ]:
if OPENAI_AVAILABLE:
    result = rag_query("What are the biggest AI-related risks disclosed by S&P 100 tech companies? Be specific.")

    print("QUESTION:", result["question"])
    print(f"\nSOURCES: {', '.join(f'{t} ({d})' for t, d, s in result['sources'])}")
    print(f"TOKENS:  {result['tokens']['input']} in, {result['tokens']['output']} out")
    print("\n" + "=" * 70)
    print(result["answer"])
else:
    print("Skipping — no OpenAI API key.")

---
## 13 — LLM-Powered Agent with Tool Use

Now we replace the **rule-based planner** from Section 5 with GPT-4o. The LLM decides autonomously which tools to call and in what order, using OpenAI's native [function calling](https://platform.openai.com/docs/guides/function-calling) API.

This is how production AI agents work at banks today:

```
         ┌─────────────────────────────────────────────────────┐
         │                   GPT-4o AGENT                      │
         │                                                     │
  Goal ──→  Think: "I need to compare JPM and GS..."          │
         │  Call:  search("JPM credit risk management")        │
         │  Call:  search("GS credit risk management")         │
         │  Think: "Now let me look at specific metrics..."    │
         │  Call:  search("JPM provision for credit losses")   │
         │  Think: "I have enough to write a synthesis."       │
         │  Answer: "Here are the key differences..."      ──→ Output
         └─────────────────────────────────────────────────────┘
```

The LLM can call tools **multiple times**, building up context iteratively — just like a human analyst would.

In [ ]:
# ── Define tools in OpenAI function-calling format ──────────────────────────
OPENAI_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_filings",
            "description": (
                "Semantic search across 10-K annual filings of 30 S&P 100 companies "
                "(Tech, Banking, Healthcare, Energy, Consumer, Industrial sectors). "
                "Returns the most relevant excerpts with company ticker, filing date, "
                "and relevance score. Use specific financial terms for better results."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Search query — use specific financial terms, e.g. 'credit loss provisions' not just 'risk'",
                    },
                    "num_results": {
                        "type": "integer",
                        "description": "Number of results to return (1-10)",
                        "default": 5,
                    },
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_filing_chunk",
            "description": (
                "Retrieve the full text of a specific chunk by ticker and chunk index. "
                "Use this after search_filings to read a full passage that looked promising."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "ticker": {"type": "string", "description": "Company ticker, e.g. 'JPM'"},
                    "chunk_index": {"type": "integer", "description": "Chunk index from search results"},
                },
                "required": ["ticker", "chunk_index"],
            },
        },
    },
]

def execute_tool(name, args):
    """Execute a tool call and return the result as a string."""
    if name == "search_filings":
        k = min(args.get("num_results", 5), 10)
        results = retrieve_oai(args["query"], k=k)
        # Enrich with chunk indices for follow-up
        enriched = []
        for r in results:
            idx = next(
                (i for i, m in enumerate(metadata)
                 if m["ticker"] == r["ticker"] and m["date"] == r["date"]
                 and chunks[i][:100] == r["text"][:100]),
                -1,
            )
            enriched.append({
                **r,
                "chunk_index": metadata[idx]["chunk_idx"] if idx >= 0 else -1,
                "text": chunks[idx][:800] if idx >= 0 else r["text"],
            })
        return json.dumps(enriched, indent=2)

    elif name == "get_filing_chunk":
        ticker = args["ticker"]
        chunk_idx = args["chunk_index"]
        for i, m in enumerate(metadata):
            if m["ticker"] == ticker and m["chunk_idx"] == chunk_idx:
                return json.dumps({
                    "ticker": ticker,
                    "date": m["date"],
                    "chunk_index": chunk_idx,
                    "full_text": chunks[i],
                })
        return json.dumps({"error": f"Chunk {chunk_idx} not found for {ticker}"})

    return json.dumps({"error": f"Unknown tool: {name}"})

print("OpenAI tools defined:", [t["function"]["name"] for t in OPENAI_TOOLS])

### 13.1 — The LLM Agent Loop

This is the core agent: a **`while` loop** that keeps calling GPT-4o until it decides it has enough information to answer. At each iteration, the model can choose to call a tool or return a final answer.

Note how similar this is to the rule-based agent from Section 5 — but now the **LLM is the planner**, choosing tools dynamically based on what it has learned so far.

In [ ]:
AGENT_SYSTEM_PROMPT = """You are a senior financial analyst AI agent with access to SEC 10-K 
annual filings from 30 major S&P 100 companies across Tech, Banking, Healthcare, Energy, 
Consumer, and Industrial sectors.

Your approach:
1. PLAN: Think about what information you need to answer the question
2. SEARCH: Use search_filings to find relevant passages (use specific financial terms)
3. DRILL DOWN: If a result looks promising, use get_filing_chunk to read the full passage
4. ITERATE: Search again with refined queries if needed
5. SYNTHESISE: Once you have enough evidence, write a clear, well-cited answer

Always cite your sources as [TICKER, filing date]. Be specific with numbers and facts.
If you can't find enough information, say so honestly."""

MAX_AGENT_TURNS = 8  # safety limit

def run_llm_agent(question):
    """Run the GPT-4o agent loop with tool use."""
    messages = [
        {"role": "system", "content": AGENT_SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    total_tokens = {"input": 0, "output": 0}
    tool_calls_log = []

    for turn in range(MAX_AGENT_TURNS):
        response = oai.chat.completions.create(
            model=LLM_MODEL,
            messages=messages,
            tools=OPENAI_TOOLS,
            temperature=0.2,
        )

        total_tokens["input"] += response.usage.prompt_tokens
        total_tokens["output"] += response.usage.completion_tokens

        msg = response.choices[0].message

        # If the model wants to call tools
        if msg.tool_calls:
            messages.append(msg)
            for tc in msg.tool_calls:
                fn_name = tc.function.name
                fn_args = json.loads(tc.function.arguments)
                tool_calls_log.append({"turn": turn + 1, "tool": fn_name, "args": fn_args})
                print(f"  Turn {turn+1}: {fn_name}({json.dumps(fn_args)[:100]}...)")

                result = execute_tool(fn_name, fn_args)
                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": result,
                })
        else:
            # Model is done — return final answer
            return {
                "question": question,
                "answer": msg.content,
                "tool_calls": tool_calls_log,
                "turns": turn + 1,
                "tokens": total_tokens,
            }

    # Hit max turns — ask for final answer
    messages.append({"role": "user", "content": "Please provide your final answer now based on what you've found."})
    response = oai.chat.completions.create(model=LLM_MODEL, messages=messages, temperature=0.2)
    total_tokens["input"] += response.usage.prompt_tokens
    total_tokens["output"] += response.usage.completion_tokens

    return {
        "question": question,
        "answer": response.choices[0].message.content,
        "tool_calls": tool_calls_log,
        "turns": MAX_AGENT_TURNS,
        "tokens": total_tokens,
    }

print("LLM agent ready.")

### 13.2 — Agent Demo: Complex multi-step analysis

Watch the agent **autonomously decide** what to search for, iterate on its queries, and synthesise a grounded answer. The tool calls are printed in real time so you can see the "thinking" process.

In [ ]:
if OPENAI_AVAILABLE:
    print("Agent working...\n")
    result = run_llm_agent(
        "Compare how tech companies (AAPL, MSFT, NVDA) and banks (JPM, GS) "
        "discuss artificial intelligence in their latest 10-K filings. "
        "How do their perspectives differ? Who sees AI more as a risk vs an opportunity?"
    )
    print(f"\nCompleted in {result['turns']} turns, {len(result['tool_calls'])} tool calls")
    print(f"Tokens: {result['tokens']['input']:,} in, {result['tokens']['output']:,} out")
    print("\n" + "=" * 70)
    print(result["answer"])
else:
    print("Skipping — no OpenAI API key.")

In [ ]:
if OPENAI_AVAILABLE:
    print("Agent working...\n")
    result = run_llm_agent(
        "Which S&P 100 companies in our database have the highest exposure to "
        "interest rate risk? What specific hedging strategies do they mention?"
    )
    print(f"\nCompleted in {result['turns']} turns, {len(result['tool_calls'])} tool calls")
    print(f"Tokens: {result['tokens']['input']:,} in, {result['tokens']['output']:,} out")
    print("\n" + "=" * 70)
    print(result["answer"])
else:
    print("Skipping — no OpenAI API key.")

---
## 14 — Final Takeaways

### What we built — two versions of the same system

| Component | Part I (Free) | Part II (OpenAI) |
|---|---|---|
| **Embeddings** | TF-IDF (25,000-dim sparse) | `text-embedding-3-small` (1,536-dim dense) |
| **Retrieval** | Keyword matching via cosine similarity | Semantic search — understands synonyms |
| **Planner** | Rule-based `if/else` | GPT-4o with function calling |
| **Answer generation** | Structured JSON | Natural-language synthesis with citations |
| **Cost** | $0 | ~$0.50 for entire notebook |

### The progression from prototype to production

```
Level 0: Ctrl+F search in PDFs             ← manual, doesn't scale
Level 1: TF-IDF + FAISS (Sections 1-8)     ← free, fast, good for keywords
Level 2: OpenAI embeddings + RAG (Sec 11-12) ← semantic understanding
Level 3: LLM agent with tools (Sec 13)      ← autonomous multi-step reasoning
Level 4: Multi-agent systems                 ← multiple specialised agents collaborate
```

Each level adds capability *and* cost. The right level depends on your use case:

| Use case | Recommended level |
|---|---|
| Simple keyword search over filings | Level 1 |
| "Find all mentions of X across 100 companies" | Level 2 |
| "Compare risk profiles of these 5 banks" | Level 3 |
| "Conduct full due diligence on this acquisition target" | Level 4 |

### Try your own questions

Modify the `run_llm_agent()` calls above with your own questions. The agent has access to the latest 10-K from all 30 loaded companies — ask it anything about their financials, risks, strategies, or competitive positioning.